# Laya — lectures batch 2 (download + speech-to-text)

This notebook fetches audio, transcribes it, and runs **engine v4** on it (engine v5 is still being built in another session). Later, engine v5 (normal and *loose* profiles) will run on the same cached transcripts.

| Lectures | Why |
|---|---|
| **YouTube**, 12 sheikhs (the 10 from batch 1 + صالح سندي and محمد سعيد رسلان), 3–4 videos each, 10–35 min | mostly hadith and text commentaries: full of quotes, narrator chains and partial echoes, which is where the "not a quote" cards come from |
| **Bin Baz**, 8 new short radio/phone talks | they have a written transcript (ground truth), so the loose profile's precision can be measured |

The list is in `backend/laya_datagen/lectures.py` (`YOUTUBE_2`, `BINBAZ_2`). Everything is cached: a re-run only fetches what is missing. **No labelling here** (no Sonnet calls).

## Setup

In [ ]:
import sys, os, json, importlib
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "backend").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from IPython.display import display
from backend.laya_datagen import config, asr as asr_mod, lectures   # config loads GROQ_API_KEY from .env
from backend.laya_datagen.sources import binbaz, youtube
importlib.reload(lectures)          # picks up list edits without restarting the kernel

RUN_DOWNLOADS = True    # True: download + transcribe what is missing (uses Groq) · False: inventory from what is already on disk
RUN_ENGINE = True      # True: engine v4 finds quotes in every transcribed lecture (local, cached) · False: skip
ENGINE = "v4"
HADITH_CORPUS = str(ROOT / "data/corpora/hadith_open_hadith_data.json")

## Download + transcribe
Each lecture is wrapped in a try/except: a failure prints `✘ <id>: <error>` and the loop goes on. After a Groq rate-limit (429) or key error, the remaining lectures are still downloaded but not transcribed: **re-run this cell later** (Groq's free tier allows only a few hours of audio per hour and per day) and it resumes where it stopped.

For Bin Baz, the word counts of the written text and of the speech are logged: they should be roughly the same size (batch 1's gate is in `pipeline.process_cached`, which runs later with the engine).

In [ ]:
errors = {}                     # doc id -> last error of this run (shown in the inventory)
groq_stop = None                # set after a rate-limit / key error: no more Groq calls this run

def transcribe(doc_id, audio):
    global groq_stop
    if groq_stop:
        raise RuntimeError(f"skipped (Groq stopped: {groq_stop})")
    try:
        return asr_mod.transcribe(audio, backend="groq", model="whisper-large-v3-turbo")
    except Exception as e:
        msg = str(e).lower()
        if "429" in msg or "rate" in msg or "quota" in msg:
            groq_stop = "rate limit"
        elif "401" in msg or "403" in msg or "invalid api key" in msg or "غير متوفر" in msg:
            groq_stop = "auth / key error: put a new GROQ_API_KEY in .env"
        raise

def fail(doc_id, e):
    errors[doc_id] = f"{type(e).__name__}: {str(e)[:150]}"
    print(f"✘ {doc_id}: {errors[doc_id]}")

if RUN_DOWNLOADS:
    for sheikh, vids in lectures.YOUTUBE_2.items():
        for vid in vids:
            doc_id = f"yt_{vid}"
            try:
                v = youtube.download(vid, sheikh)
                a = transcribe(doc_id, youtube.AUDIO_DIR / f"{v.id}.mp3")
                youtube.save_transcript(v.id, a)
                print(f"✔ {sheikh} {vid}: {v.duration / 60:.0f} min, {len(a['words'])} words")
            except Exception as e:
                fail(doc_id, e)
    for url in lectures.BINBAZ_2:
        try:
            les = binbaz.get_lesson(url)
            a = transcribe(les.id, binbaz.download_mp3(les))
            youtube.save_transcript(les.id, a, {"id": les.id, "url": les.url, "title": les.title, "sheikh": "bin_baz"})
            n_text, n_asr = sum(len(b.text.split()) for b in les.blocks), len(a["words"])
            flag = "" if 0.6 <= n_asr / max(n_text, 1) <= 1.6 else "   ⚠ sizes differ a lot: check text and audio match"
            print(f"✔ {les.id}: written {n_text} words · speech {n_asr} words (×{n_asr / max(n_text, 1):.2f}){flag}")
        except Exception as e:
            fail(binbaz.parse_lesson("", url).id, e)
    if groq_stop:
        print(f"\nGroq stopped this run ({groq_stop}). Audio is downloaded; re-run this cell later to transcribe the rest.")

## Inventory
Built from what is on disk, so it is also right with `RUN_DOWNLOADS = False`.

In [ ]:
errors = globals().get("errors", {})
rows = []

def words_of(doc_id):
    p = youtube.LECTURES_DIR / doc_id / "words.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None

def status(doc_id, audio, words):
    if words is not None: return "transcribed"
    if audio.exists(): return "audio only" + (f" ({errors[doc_id][:60]})" if doc_id in errors else "")
    return errors.get(doc_id, "not downloaded")[:80]

for sheikh, vids in lectures.YOUTUBE_2.items():
    for vid in vids:
        doc_id = f"yt_{vid}"
        meta_p = youtube.LECTURES_DIR / doc_id / "meta.json"
        meta = json.loads(meta_p.read_text(encoding="utf-8")) if meta_p.exists() else {}
        w = words_of(doc_id)
        rows.append({"source": "YouTube", "sheikh": sheikh, "id": doc_id, "title": meta.get("title", "")[:60],
                     "minutes": round(meta.get("duration", 0) / 60, 1), "ASR words": len(w) if w else 0,
                     "status": status(doc_id, youtube.AUDIO_DIR / f"{doc_id}.mp3", w)})
for url in lectures.BINBAZ_2:
    les = binbaz.parse_lesson("", url)                       # id from the URL, no request
    try: les = binbaz.get_lesson(url)                        # title from the cached page
    except Exception: pass
    w = words_of(les.id)
    rows.append({"source": "Bin Baz", "sheikh": "bin_baz", "id": les.id, "title": les.title[:60],
                 "minutes": round(w[-1]["end"] / 60, 1) if w else 0.0, "ASR words": len(w) if w else 0,
                 "status": status(les.id, youtube.AUDIO_DIR / f"{les.id}.mp3", w)})

inv = pd.DataFrame(rows)
pd.set_option("display.max_rows", 100, "display.max_colwidth", 70)
display(inv)
inv["done"] = inv.status.eq("transcribed")
per = inv.groupby("sheikh", sort=False).agg(lectures=("id", "size"), transcribed=("done", "sum"),
                                            minutes=("minutes", "sum"), asr_words=("ASR words", "sum"))
display(per.round(1))
done = inv[inv.done]
print(f"{len(done)} / {len(inv)} lectures transcribed · {done.minutes.sum() / 60:.1f} h of new audio transcribed "
      f"({inv.minutes.sum() / 60:.1f} h known so far; Bin Baz minutes are only known once transcribed)")

## Engine v4 findings
Engine v4 runs on every transcribed lecture (same calls as batch 1 in `laya_autolabel.ipynb`, cached on the engine's fingerprint). Bin Baz lessons whose written text and speech don't match well enough are dropped, as in batch 1. Nothing is labelled here.

In [ ]:
bundles = {}
if RUN_ENGINE:
    from backend.laya_datagen import pipeline, autolabel
    from backend.laya_datagen.engine_bridge import EngineBridge
    from backend.laya_datagen.cache import JsonCache
    bridge = EngineBridge(hadith_path=HADITH_CORPUS); bridge.start()
    asr_cache = JsonCache("asr")

    def cached_asr(audio):          # only what is already transcribed: no Groq call from this cell
        return asr_cache.get(asr_mod.sha(asr_mod._file_sha(audio), "groq", "whisper-large-v3-turbo"))

    for sheikh, vids in lectures.YOUTUBE_2.items():
        for vid in vids:
            doc_id = f"yt_{vid}"
            audio = youtube.AUDIO_DIR / f"{doc_id}.mp3"
            a = cached_asr(audio) if audio.exists() else None
            if a is None: continue
            try:
                meta = json.loads((youtube.LECTURES_DIR / doc_id / "meta.json").read_text(encoding="utf-8"))
                bundles[doc_id] = pipeline.process_no_gt(doc_id, meta["title"], meta["url"], a, bridge, [ENGINE])
            except Exception as e:
                print(f"✘ {doc_id}: {type(e).__name__}: {str(e)[:150]}")
    for url in lectures.BINBAZ_2:
        les = binbaz.get_lesson(url)
        audio = youtube.AUDIO_DIR / f"{les.id}.mp3"
        a = cached_asr(audio) if audio.exists() else None
        if a is None: continue
        try:
            b = pipeline.process_cached(les, a, bridge, [ENGINE])
        except Exception as e:
            print(f"✘ {les.id}: {type(e).__name__}: {str(e)[:150]}"); continue
        if not b["report"]["passed_gate"]:
            print(f"✘ {les.id}: written text and speech don't match well enough"); continue
        bundles[les.id] = b

    inv["findings (v4)"] = inv.id.map(lambda i: len(bundles[i]["records"]) if i in bundles else None)
    inv["on a GT quote"] = inv.id.map(lambda i: sum(autolabel.is_gt_finding(r) for r in bundles[i]["records"])
                                      if i in bundles and i.startswith("binbaz") else None)
    display(inv[["source", "sheikh", "id", "title", "minutes", "findings (v4)", "on a GT quote"]])
    n = sum(len(b["records"]) for b in bundles.values())
    n_gt = int(inv["on a GT quote"].fillna(0).sum())
    print(f"engine {ENGINE}: {len(bundles)} lectures · {n} findings · {n - n_gt} to label (not on a quote marked in Bin Baz's text)")

## Next step
1. **Sonnet labels the v4 findings** of batch 2 (as in `laya_autolabel.ipynb`). For Bin Baz, findings on a quote marked in the written text get their label automatically.
2. When **engine v5** is ready: run it with both the normal and the *loose* profile on batch 1 (`lectures.YOUTUBE`, `lectures.BINBAZ`) and batch 2 (`YOUTUBE_2`, `BINBAZ_2`). The transcripts are cached, so this costs no Groq calls.